# SpectraLite — Research Lab (`works.ipynb`)

Open via **File → Open notebook → GitHub** (do not double-click in Files).

## Rule

Each **PHASE N RUN** cell is **self-contained**:
sync git → deps → load model → run phase → write `results/`.

You can run **only that phase**. Use `FORCE_RERUN_PHASEN = True` to remeasure.

| Phase | In git? | Headline |
|-------|---------|----------|
| 0 | Yes | OPT-125M loads on A100; 73 Linears |
| 1 | Yes | Dense ~7.2ms prefill, ~7.1ms/tok; C4 PPL~28.7 (set FORCE_RERUN_PHASE1 for WT2/PTB) |
| 2 | Yes | Vanilla SVD: PPL explodes; latency **slower** than dense |
| 3 | **Runnable now** | Activation-aware SVD (whitening) |
| 4 | **Runnable now** | SpectraLite spectral-entropy allocation |
| 5 | **Runnable now** | Ledoit-Wolf + κ gating |
| 6 | **Runnable now** | Latency gate + speedup |
| 7+ | Not yet | Ablations next |


---
# Phase 0 — Environment & Smoke Test


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE0 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.artifacts import (
    is_phase_complete, save_phase0_artifacts, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.system import print_environment_report
from spectralite.model_loader import print_model_load_summary, generate_text
from spectralite.model_analysis import print_full_model_analysis
from spectralite import gpu
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("0", cfg) and not FORCE_RERUN_PHASE0:
    print_section("Phase 0 cached in results/ — skip (set FORCE_RERUN_PHASE0=True to redo)")
    print_progress_dashboard(cfg)
else:
    env_info = print_environment_report()
    mem_before = gpu.snapshot(label="Memory before loading")
    load_summary = print_model_load_summary(model, model_name=cfg.model_name)
    mem_after_load = gpu.snapshot(label="Memory after loading")
    analysis = print_full_model_analysis(model, model_name=cfg.model_name)
    generated = generate_text(model, tokenizer, cfg.smoke_prompt, max_new_tokens=cfg.max_new_tokens)
    print_section("Test Inference"); print(generated)
    mem_after_infer = gpu.snapshot(label="Memory after inference")
    gpu.print_memory_timeline([mem_before, mem_after_load, mem_after_infer])
    inference_ok = bool(generated and generated.strip())
    ok = inference_ok and gpu.is_cuda_available()
    print_checklist([
        ("GPU Ready", gpu.is_cuda_available()),
        ("Model Loaded", model is not None),
        ("Inference OK", inference_ok),
        ("Phase 0 Complete", ok),
    ])
    assert ok
    save_phase0_artifacts(env_info=env_info, load_summary=load_summary, analysis=analysis,
                         inference_ok=inference_ok, config=cfg)
    print_git_save_instructions()


---
# Phase 1 — Dense Baseline

Set `FORCE_RERUN_PHASE1 = True` once to refresh WT2/PTB PPL (first run had NaNs; code is fixed).


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE1 = True  # True => remeasure dense baseline with fixed PPL

from pathlib import Path
from spectralite.phase_runner import bootstrap_phase
from spectralite.benchmark import run_phase1_dense_baseline
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.results_io import load_results
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("1", cfg) and not FORCE_RERUN_PHASE1:
    print_section("Phase 1 cached — loading results/phase1_summary.json")
    row = read_json("phase1_summary.json", cfg).get("row", {})
else:
    out = run_phase1_dense_baseline(
        model, tokenizer, config=cfg,
        prompt_len=getattr(cfg, "latency_prompt_len", 128),
        gen_tokens=getattr(cfg, "latency_gen_tokens", 64),
        latency_warmup=getattr(cfg, "latency_warmup", 10),
        latency_reps_prefill=getattr(cfg, "latency_reps_prefill", 50),
        latency_reps_decode=getattr(cfg, "latency_reps_decode", 30),
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=getattr(cfg, "ppl_max_tokens", 50_000),
        run_calflops=True, run_ppl=True,
        csv_name="phase1_dense_baselines.csv",
        phase="1", method="dense", persist_artifacts=True,
    )
    row = out["row"]

ok = row.get("empirical_flops_fwd") is not None and row.get("prefill_ms_mean") is not None
print_checklist([
    ("FLOPs", row.get("empirical_flops_fwd") is not None),
    ("Prefill", row.get("prefill_ms_mean") is not None),
    ("Decode", row.get("decode_ms_per_token_mean") is not None),
    ("C4 PPL", row.get("ppl_c4") not in (None, "")),
    ("WT2 PPL", row.get("ppl_wikitext2") not in (None, "", "nan")),
    ("Phase 1 Complete", ok and is_phase_complete("1", cfg)),
])
print_section("Phase 1 Outcome")
for k in ["prefill_ms_mean","decode_ms_per_token_mean","throughput_tokens_per_s",
          "ppl_wikitext2","ppl_ptb","ppl_c4","empirical_flops_fwd"]:
    print(f"  {k}: {row.get(k)}")
print(load_results(Path(cfg.results_dir)/"phase1_dense_baselines.csv").tail(2).to_string(index=False))
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


Synced: /content/Execution/SpectraLite

 Repo synced
  PACKAGE_ROOT = /content/Execution/SpectraLite
  phase2.py exists = True
  phase3.py exists = True
SpectraLite environment ready
  repo_root      : /content/Execution/SpectraLite
  requirements   : /content/Execution/SpectraLite/requirements-colab.txt
  torch          : 2.11.0+cu128
  cuda_available : True
  gpu            : NVIDIA A100-SXM4-40GB

 SpectraLite progress (git-tracked artifacts)
  Status file                  /content/Execution/SpectraLite/results/phase_status.json
  Updated at                   2026-07-12T16:29:16Z
  Phase 0  [DONE]  2026-07-12T16:06:46Z
           - summary: results/phase0_summary.json
           - linear_layers: results/phase0_linear_layers.json
  Phase 1  [DONE]  2026-07-12T16:08:32Z
           - summary: results/phase1_summary.json
           - csv: results/phase1_dense_baselines.csv
           - status: results/phase_status.json
  Phase 2  [DONE]  2026-07-12T16:29:16Z
           - summary: result

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

2026-07-12 16:47:09 | INFO     | spectralite.perplexity | wikitext2 ppl=43.9631 over 49902 tokens
2026-07-12 16:47:09 | INFO     | spectralite.perplexity | Computing perplexity: ptb (seq_len=512, max_tokens=50000)
2026-07-12 16:47:10 | WARNING  | spectralite.perplexity | PPL failed for ptb: Dataset 'penn_treebank' doesn't exist on the Hub or cannot be accessed.
2026-07-12 16:47:10 | INFO     | spectralite.perplexity | Computing perplexity: c4 (seq_len=512, max_tokens=50000)


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-07-12 16:47:15 | INFO     | spectralite.perplexity | c4 ppl=28.7403 over 49902 tokens
  WikiText-2 PPL               43.963069915771484
  PTB PPL                      nan
  C4 PPL                       28.740310668945312

 Results written
  CSV                          /content/Execution/SpectraLite/results/phase1_dense_baselines.csv
2026-07-12 16:47:15 | INFO     | spectralite.artifacts | Marked phase 1 complete

 Phase 1 artifacts saved (git these files)
  summary                      results/phase1_summary.json
  csv                          results/phase1_dense_baselines.csv
  status                       results/phase_status.json

 Phase 0 Status
  [PASS] FLOPs
  [PASS] Prefill
  [PASS] Decode
  [PASS] C4 PPL
  [PASS] WT2 PPL
  [PASS] Phase 1 Complete

 Phase 1 Outcome
  prefill_ms_mean: 7.2976383781433105
  decode_ms_per_token_mean: 7.238991053899129
  throughput_tokens_per_s: 138.14079787560604
  ppl_wikitext2: 43.963069915771484
  ppl_ptb: nan
  ppl_c4: 28.740310668945312


---
# Phase 2 — Vanilla Truncated SVD

Cached in git: r=0.5/0.4/0.3 → C4 PPL ~922/2953/6256 vs dense ~28.7;  
decode ~9.8 ms/tok vs dense ~7.1 (**no speedup**). Set `FORCE_RERUN_PHASE2 = True` to redo.


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE2 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.phase2 import run_phase2_vanilla_svd_sweep
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("2", cfg) and not FORCE_RERUN_PHASE2:
    print_section("Phase 2 cached — loading results/phase2_summary.json")
    phase2 = read_json("phase2_summary.json", cfg)
else:
    phase2 = run_phase2_vanilla_svd_sweep(
        model, tokenizer, config=cfg,
        rank_ratios=(0.5, 0.4, 0.3),
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=30_000,
        latency_reps_prefill=30, latency_reps_decode=20,
        csv_name="phase2_vanilla_svd.csv",
    )

rows = phase2.get("rows", [])
ok = is_phase_complete("2", cfg) and len(rows) > 0
print_checklist([
    ("Artifacts", is_phase_complete("2", cfg)),
    ("Rows", len(rows) > 0),
    ("Phase 2 Complete", ok),
])
print_section("Phase 2 Outcome")
for row in rows:
    print(f"  {row.get('method')}: prefill={row.get('prefill_ms_mean')} decode={row.get('decode_ms_per_token_mean')} C4={row.get('ppl_c4')}")
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


---
# Phase 3 — Activation-Aware SVD (ASVD / SVD-LLM style)

**Self-contained RUN cell** below.

**What it does (plan Phase 3):**
1. WikiText-2 calibration forwards
2. Hook Linear inputs → covariance ``C = XᵀX/n + λI``
3. Cholesky whitening → truncated SVD on ``W L`` → fuse factors → ``LowRankLinear``
4. Same rank ratios as Phase 2 (`0.5, 0.4, 0.3`) for fair comparison
5. Measure PPL / latency / FLOPs; write `results/phase3_*`

**Compare to Phase 2:** whitening should improve PPL at matched rank ratios (still may not speed up — latency gate is Phase 6).

Set `FORCE_RERUN_PHASE3 = True` to remeasure.


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE3 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.phase3 import run_phase3_activation_aware_sweep
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("3", cfg) and not FORCE_RERUN_PHASE3:
    print_section("Phase 3 cached — loading results/phase3_summary.json")
    phase3 = read_json("phase3_summary.json", cfg)
else:
    phase3 = run_phase3_activation_aware_sweep(
        model,
        tokenizer,
        config=cfg,
        rank_ratios=(0.5, 0.4, 0.3),
        calib_num_sequences=getattr(cfg, "calib_num_sequences", 32),
        calib_seq_len=min(getattr(cfg, "calib_seq_len", 2048), 512),  # Colab-speed default
        calib_batch_size=2,
        ridge=1e-2,
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=30_000,
        latency_reps_prefill=30,
        latency_reps_decode=20,
        csv_name="phase3_activation_aware_svd.csv",
    )

rows = phase3.get("rows", [])
ok = is_phase_complete("3", cfg) and len(rows) > 0
print_checklist([
    ("Artifacts", is_phase_complete("3", cfg)),
    ("Rows", len(rows) > 0),
    ("Phase 3 Complete", ok),
])
print_section("Phase 3 Outcome (vs remember Phase 2 C4 ~922/2953/6256)")
for row in rows:
    print(
        f"  {row.get('method')}: prefill={row.get('prefill_ms_mean')} "
        f"decode={row.get('decode_ms_per_token_mean')} "
        f"C4={row.get('ppl_c4')} WT2={row.get('ppl_wikitext2')}"
    )
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


Synced: /content/Execution/SpectraLite

 Repo synced
  PACKAGE_ROOT = /content/Execution/SpectraLite
  phase2.py exists = True
  phase3.py exists = True
SpectraLite environment ready
  repo_root      : /content/Execution/SpectraLite
  requirements   : /content/Execution/SpectraLite/requirements-colab.txt
  torch          : 2.11.0+cu128
  cuda_available : True
  gpu            : NVIDIA A100-SXM4-40GB

 SpectraLite progress (git-tracked artifacts)
  Status file                  /content/Execution/SpectraLite/results/phase_status.json
  Updated at                   2026-07-12T16:29:16Z
  Phase 0  [DONE]  2026-07-12T16:06:46Z
           - summary: results/phase0_summary.json
           - linear_layers: results/phase0_linear_layers.json
  Phase 1  [DONE]  2026-07-12T16:08:32Z
           - summary: results/phase1_summary.json
           - csv: results/phase1_dense_baselines.csv
           - status: results/phase_status.json
  Phase 2  [DONE]  2026-07-12T16:29:16Z
           - summary: result

RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_mm)

---
# Phase 4 — SpectraLite Spectral Rank Allocation (novelty)

**Self-contained RUN cell** below.

**What it does (plan Phase 4):**
1. WikiText-2 calibration + activation whitening (same as Phase 3)
2. Per-matrix Roy–Vetterli ``ρ_eff = exp(H)`` on whitened singular values
3. Spectral compressibility ``s = 1 − ρ_eff/q`` + stable-rank importance
4. Protect score → global binary-search ``λ`` under **Phase-3-matched FLOP keep ratios**
5. Measure PPL / latency; write `results/phase4_*`

**Success bar:** beat Phase 3 C4 PPL (~123 / 555 / 2287) at the same keep ratios.

Set `FORCE_RERUN_PHASE4 = True` to remeasure.


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE4 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.phase4 import run_phase4_spectralite_sweep
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("4", cfg) and not FORCE_RERUN_PHASE4:
    print_section("Phase 4 cached — loading results/phase4_summary.json")
    phase4 = read_json("phase4_summary.json", cfg)
else:
    phase4 = run_phase4_spectralite_sweep(
        model,
        tokenizer,
        config=cfg,
        target_keep_ratios=None,  # auto from phase3_summary.json
        calib_num_sequences=getattr(cfg, "calib_num_sequences", 32),
        calib_seq_len=min(getattr(cfg, "calib_seq_len", 2048), 512),
        calib_batch_size=2,
        ridge=getattr(cfg, "whitening_ridge", 1e-2),
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=30_000,
        latency_reps_prefill=30,
        latency_reps_decode=20,
        csv_name="phase4_spectralite.csv",
    )

rows = phase4.get("rows", [])
ok = is_phase_complete("4", cfg) and len(rows) > 0
print_checklist([
    ("Artifacts", is_phase_complete("4", cfg)),
    ("Rows", len(rows) > 0),
    ("Phase 4 Complete", ok),
])
print_section("Phase 4 Outcome (vs Phase 3 C4 ~123 / 555 / 2287)")
for row in rows:
    print(
        f"  {row.get('method')}: keep={row.get('analytic_flops_fwd_ratio')} "
        f"prefill={row.get('prefill_ms_mean')} "
        f"decode={row.get('decode_ms_per_token_mean')} "
        f"C4={row.get('ppl_c4')} WT2={row.get('ppl_wikitext2')}"
    )
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


Synced: /content/Execution/SpectraLite

 Repo synced
  PACKAGE_ROOT = /content/Execution/SpectraLite
  phase2.py exists = True
  phase3.py exists = True
  phase4.py exists = True
SpectraLite environment ready
  repo_root      : /content/Execution/SpectraLite
  requirements   : /content/Execution/SpectraLite/requirements-colab.txt
  torch          : 2.11.0+cu128
  cuda_available : True
  gpu            : NVIDIA A100-SXM4-40GB

 SpectraLite progress (git-tracked artifacts)
  Status file                  /content/Execution/SpectraLite/results/phase_status.json
  Updated at                   2026-07-12T16:58:54Z
  Phase 0  [DONE]  2026-07-12T16:06:46Z
           - summary: results/phase0_summary.json
           - linear_layers: results/phase0_linear_layers.json
  Phase 1  [DONE]  2026-07-12T16:47:15Z
           - summary: results/phase1_summary.json
           - csv: results/phase1_dense_baselines.csv
           - status: results/phase_status.json
  Phase 2  [DONE]  2026-07-12T16:29:16Z
  

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-07-12 17:10:11 | INFO     | spectralite.perplexity | c4 ppl=4798.6157 over 29941 tokens
  WikiText-2 PPL               3139.700927734375
  PTB PPL                      nan
  C4 PPL                       4798.61572265625

 Results written
  CSV                          /content/Execution/SpectraLite/results/phase4_spectralite.csv
2026-07-12 17:10:11 | INFO     | spectralite.phase4 | SpectraLite target_keep=0.6000
2026-07-12 17:10:11 | INFO     | spectralite.rank_alloc | Allocated ranks: target_keep=0.6000 got=0.5998 lambda=69.750000 layers=72
2026-07-12 17:10:11 | INFO     | spectralite.svd_spectralite | SpectraLite model.decoder.layers.0.self_attn.k_proj: (768×768)→r=116 (rho=242.0 protect=0.0022) params 590592→178944
2026-07-12 17:10:11 | INFO     | spectralite.svd_spectralite | SpectraLite model.decoder.layers.0.self_attn.v_proj: (768×768)→r=426 (rho=476.5 protect=0.0079) params 590592→655104
2026-07-12 17:10:11 | INFO     | spectralite.svd_spectralite | SpectraLite model.decode

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-07-12 17:10:38 | INFO     | spectralite.perplexity | c4 ppl=7257.7476 over 29941 tokens
  WikiText-2 PPL               5869.36474609375
  PTB PPL                      nan
  C4 PPL                       7257.74755859375

 Results written
  CSV                          /content/Execution/SpectraLite/results/phase4_spectralite.csv
2026-07-12 17:10:38 | INFO     | spectralite.phase4 | SpectraLite target_keep=0.4498
2026-07-12 17:10:38 | INFO     | spectralite.rank_alloc | Allocated ranks: target_keep=0.4498 got=0.4493 lambda=35.750000 layers=72
2026-07-12 17:10:38 | INFO     | spectralite.svd_spectralite | SpectraLite model.decoder.layers.0.self_attn.k_proj: (768×768)→r=59 (rho=242.0 protect=0.0022) params 590592→91392
2026-07-12 17:10:38 | INFO     | spectralite.svd_spectralite | SpectraLite model.decoder.layers.0.self_attn.v_proj: (768×768)→r=218 (rho=476.5 protect=0.0079) params 590592→335616
2026-07-12 17:10:38 | INFO     | spectralite.svd_spectralite | SpectraLite model.decoder.l

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-07-12 17:11:04 | INFO     | spectralite.perplexity | c4 ppl=5780.8711 over 29941 tokens
  WikiText-2 PPL               4629.02099609375
  PTB PPL                      nan
  C4 PPL                       5780.87109375

 Results written
  CSV                          /content/Execution/SpectraLite/results/phase4_spectralite.csv
2026-07-12 17:11:04 | INFO     | spectralite.artifacts | Marked phase 4 complete

 Save progress to GitHub (do this after each phase)
  In Colab (from SpectraLite root):

    %cd /content/Execution
    !git add SpectraLite/results/*.json SpectraLite/results/*.csv SpectraLite/notebooks/works.ipynb
    !git config user.email "you@example.com"
    !git config user.name "Your Name"
    !git commit -m "Record Phase N results"
    !git push

  Or: File → Save a copy in GitHub for the notebook, AND commit the results/ files.

  Next Colab session:
    1) git pull
    2) Run Stage 0 (deps)
    3) Run Session Restore — completed phases are skipped
    4) Continue from 

---
# Phase 5 — Stability Safeguards (Ledoit–Wolf + κ)

**Self-contained RUN cell** below.

**What it does (plan Phase 5):**
1. Same WikiText-2 calibration / activation hooks
2. Replace ridge covariance with **Ledoit–Wolf** shrinkage
3. **κ gate**: bump rank until ``σ₁/σ_r ≤ κ_max`` (default ``1e4``)
4. Log per-layer κ / reconstruction error
5. Evaluate:
   - **ActSVD + LW + κ** at ratios `0.5/0.4/0.3` (vs Phase 3 ~123/555/2287)
   - **SpectraLite + LW + κ** at Phase-3-matched keeps (vs Phase 4 ~4800/7258/5781)

Set `FORCE_RERUN_PHASE5 = True` to remeasure.


In [55]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE5 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.phase5 import run_phase5_stability_sweep
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("5", cfg) and not FORCE_RERUN_PHASE5:
    print_section("Phase 5 cached — loading results/phase5_summary.json")
    phase5 = read_json("phase5_summary.json", cfg)
else:
    phase5 = run_phase5_stability_sweep(
        model,
        tokenizer,
        config=cfg,
        rank_ratios=(0.5, 0.4, 0.3),
        target_keep_ratios=None,
        calib_num_sequences=getattr(cfg, "calib_num_sequences", 32),
        calib_seq_len=min(getattr(cfg, "calib_seq_len", 2048), 512),
        calib_batch_size=2,
        kappa_max=getattr(cfg, "kappa_max", 1e4),
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=30_000,
        latency_reps_prefill=30,
        latency_reps_decode=20,
        csv_name="phase5_stability.csv",
        run_actsvd=True,
        run_spectralite=True,
    )

rows = phase5.get("rows", [])
ok = is_phase_complete("5", cfg) and len(rows) > 0
print_checklist([
    ("Artifacts", is_phase_complete("5", cfg)),
    ("Rows", len(rows) > 0),
    ("Phase 5 Complete", ok),
])
print_section("Phase 5 Outcome")
print("  Refs: Phase3 C4 ~123/555/2287 | Phase4 C4 ~4800/7258/5781")
for row in rows:
    print(
        f"  {row.get('method')}: keep={row.get('analytic_flops_fwd_ratio')} "
        f"C4={row.get('ppl_c4')} WT2={row.get('ppl_wikitext2')} "
        f"decode={row.get('decode_ms_per_token_mean')}"
    )
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


Synced: /content/Execution/SpectraLite

 Repo synced
  PACKAGE_ROOT = /content/Execution/SpectraLite
  phase2.py exists = True
  phase3.py exists = True
  phase4.py exists = True
  phase5.py exists = True
SpectraLite environment ready
  repo_root      : /content/Execution/SpectraLite
  requirements   : /content/Execution/SpectraLite/requirements-colab.txt
  torch          : 2.11.0+cu128
  cuda_available : True
  gpu            : NVIDIA A100-SXM4-40GB

 SpectraLite progress (git-tracked artifacts)
  Status file                  /content/Execution/SpectraLite/results/phase_status.json
  Updated at                   2026-07-12T17:11:04Z
  Phase 0  [DONE]  2026-07-12T16:06:46Z
           - summary: results/phase0_summary.json
           - linear_layers: results/phase0_linear_layers.json
  Phase 1  [DONE]  2026-07-12T16:47:15Z
           - summary: results/phase1_summary.json
           - csv: results/phase1_dense_baselines.csv
           - status: results/phase_status.json
  Phase 2  [DONE

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-07-12 17:21:35 | INFO     | spectralite.perplexity | c4 ppl=1472.0458 over 29941 tokens
  WikiText-2 PPL               1423.076416015625
  PTB PPL                      nan
  C4 PPL                       1472.0457763671875

 Results written
  CSV                          /content/Execution/SpectraLite/results/phase5_stability.csv
2026-07-12 17:21:35 | INFO     | spectralite.whitening | Ledoit–Wolf cov: dim=768 n=8192 shrinkage=0.011766960844397545 kappa=3.7e+03
2026-07-12 17:21:35 | INFO     | spectralite.svd_activation | ActSVD model.decoder.layers.0.self_attn.k_proj: (768×768)→r=307 | acts=16384 | params 590592→472320 | kappa_cov=3.7e+03 recon=0.207
2026-07-12 17:21:35 | INFO     | spectralite.whitening | Ledoit–Wolf cov: dim=768 n=8192 shrinkage=0.011766960844397545 kappa=3.7e+03
2026-07-12 17:21:35 | INFO     | spectralite.svd_activation | ActSVD model.decoder.layers.0.self_attn.v_proj: (768×768)→r=307 | acts=16384 | params 590592→472320 | kappa_cov=3.7e+03 recon=0.421
2026-07-

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-07-12 17:27:48 | INFO     | spectralite.perplexity | c4 ppl=3452.0376 over 29941 tokens
  WikiText-2 PPL               4705.34033203125
  PTB PPL                      nan
  C4 PPL                       3452.03759765625

 Results written
  CSV                          /content/Execution/SpectraLite/results/phase5_stability.csv
2026-07-12 17:27:48 | INFO     | spectralite.rank_alloc | Allocated ranks: target_keep=0.4498 got=0.4501 lambda=20.500000 layers=72
2026-07-12 17:27:48 | INFO     | spectralite.svd_spectralite | SpectraLite model.decoder.layers.0.self_attn.k_proj: (768×768)→r=65 (rho=238.7 protect=0.0041) params 590592→100608 recon=0.664
2026-07-12 17:27:48 | INFO     | spectralite.svd_spectralite | SpectraLite model.decoder.layers.0.self_attn.v_proj: (768×768)→r=111 (rho=452.9 protect=0.0070) params 590592→171264 recon=0.759
2026-07-12 17:27:48 | INFO     | spectralite.svd_spectralite | SpectraLite model.decoder.layers.0.self_attn.q_proj: (768×768)→r=91 (rho=271.7 protect=0.

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-07-12 17:28:15 | INFO     | spectralite.perplexity | c4 ppl=3525.3269 over 29941 tokens
  WikiText-2 PPL               4828.1552734375
  PTB PPL                      nan
  C4 PPL                       3525.326904296875

 Results written
  CSV                          /content/Execution/SpectraLite/results/phase5_stability.csv
2026-07-12 17:28:15 | INFO     | spectralite.artifacts | Marked phase 5 complete

 Save progress to GitHub (do this after each phase)
  In Colab (from SpectraLite root):

    %cd /content/Execution
    !git add SpectraLite/results/*.json SpectraLite/results/*.csv SpectraLite/notebooks/works.ipynb
    !git config user.email "you@example.com"
    !git config user.name "Your Name"
    !git commit -m "Record Phase N results"
    !git push

  Or: File → Save a copy in GitHub for the notebook, AND commit the results/ files.

  Next Colab session:
    1) git pull
    2) Run Stage 0 (deps)
    3) Run Session Restore — completed phases are skipped
    4) Continue from

---
# Phase 6 — Latency Gate + Real Speedup

**Self-contained RUN cell** below.

**What it does (plan Phase 6):**
1. Same-session **dense** latency/PPL reference
2. Ridge **ActSVD ungated** at `0.5/0.4/0.3` (Phase-3 quality path)
3. **Latency gate** ``r < κ_speed · mn/(m+n)`` — keep layer dense if not feasible
4. Report prefill/decode **speedup vs dense** + C4 PPL
5. Probe **CUDA-graph** fixed-shape prefill (falls back if capture fails)
6. Factor fusion already in ``LowRankLinear`` (σ absorbed)

Expected: at ratio 0.5 many square attn layers stay dense (break-even ≈ 384).

Set `FORCE_RERUN_PHASE6 = True` to remeasure.


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path("/content/Execution")
PKG = ROOT / "SpectraLite"
REPO = "https://github.com/PrabinDevkota/Execution.git"

if not (PKG / "spectralite").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
else:
    subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"])

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

for _name in list(sys.modules):
    if _name == "spectralite" or _name.startswith("spectralite."):
        del sys.modules[_name]

print("Synced:", PKG)

FORCE_RERUN_PHASE6 = False

from spectralite.phase_runner import bootstrap_phase
from spectralite.phase6 import run_phase6_latency_gate_sweep
from spectralite.artifacts import (
    is_phase_complete, read_json, print_git_save_instructions, print_progress_dashboard,
)
from spectralite.utils import print_checklist, print_section

boot = bootstrap_phase(
    hard_reset=False, install_deps=True, require_cuda=True,
    model=globals().get("model"), tokenizer=globals().get("tokenizer"),
)
cfg, model, tokenizer = boot["cfg"], boot["model"], boot["tokenizer"]

if is_phase_complete("6", cfg) and not FORCE_RERUN_PHASE6:
    print_section("Phase 6 cached — loading results/phase6_summary.json")
    phase6 = read_json("phase6_summary.json", cfg)
else:
    phase6 = run_phase6_latency_gate_sweep(
        model,
        tokenizer,
        config=cfg,
        rank_ratios=(0.5, 0.4, 0.3),
        kappa_speed=getattr(cfg, "kappa_speed", 1.0),
        calib_num_sequences=getattr(cfg, "calib_num_sequences", 32),
        calib_seq_len=min(getattr(cfg, "calib_seq_len", 2048), 512),
        calib_batch_size=2,
        ridge=getattr(cfg, "whitening_ridge", 1e-2),
        ppl_seq_len=getattr(cfg, "ppl_seq_len", 512),
        ppl_max_tokens=30_000,
        latency_reps_prefill=30,
        latency_reps_decode=20,
        csv_name="phase6_latency_gate.csv",
        try_cuda_graph=True,
    )

rows = phase6.get("rows", [])
logs = phase6.get("gate_logs", [])
ok = is_phase_complete("6", cfg) and len(rows) > 0
print_checklist([
    ("Artifacts", is_phase_complete("6", cfg)),
    ("Rows", len(rows) > 0),
    ("Phase 6 Complete", ok),
])
print_section("Phase 6 Outcome (speedup > 1.0 means faster than dense)")
print("  CUDA-graph:", phase6.get("cuda_graph"))
for g in logs:
    print(
        f"  {g.get('method')}: keep={g.get('param_keep_ratio_touched'):.3f} "
        f"gated_dense={g.get('num_gated_dense')} "
        f"prefill_x={g.get('prefill_speedup_vs_dense'):.3f} "
        f"decode_x={g.get('decode_speedup_vs_dense'):.3f} "
        f"C4={g.get('ppl_c4')}"
    )
print_progress_dashboard(cfg)
print_git_save_instructions()
assert ok


---
# Phases 7–8 (later)

Each will be one self-contained RUN cell when implemented.


In [ ]:
print("Phases 7–8 not implemented yet.")


---
# Save results

Download `SpectraLite/results/` → PC → tell Cursor **uploaded**.
